# Appendix: word2vec similarity features (Model 1 experiment)

Builds word2vec versions of the two query-relevance features and compares them with the existing TF-IDF versions:

| New feature | Compares | TF-IDF counterpart |
|---|---|---|
| `w2v_name_sim` | slate query vs candidate's product name | `query_candidate_name_sim` |
| `w2v_cate4_sim` | slate query vs candidate's L4 category | `query_candidate_cate4_sim` |

**Why.** TF-IDF only rewards exact shared terms. Word2vec learns term vectors from co-occurrence, so related terms can match even with no word in common. The interim report planned embeddings; this notebook tests whether they beat TF-IDF.

**Corpus.** Product names only (one name = one sentence of term IDs), from the whole catalogue. Names carry no labels, so every product can be used. Queries are not needed for training: 99.5% of val query tokens appear in product names, so query vectors are built from term vectors learned from names. This also matches the TF-IDF features, which learned their term statistics from names only.

**Plan.**
1. Write the corpus once, in random order. The full file has all ~12M names; the first `SAMPLE_N` lines form a uniform random sample.
2. **Tuning:** train a few `window` settings on the sample and keep the best by per-slate NDCG@10 on **val**.
3. **Final:** retrain the chosen setting on the full catalogue, build both features for train/val/test and save them.
4. Evaluate on val: per-slate metrics vs TF-IDF and random, the zero-TF-IDF subset, and correlation with TF-IDF. Section 9 records the in-model comparison (TF-IDF vs word2vec vs both).

**Before running:** add a shortcut to the shared `Grp8Submission` folder in your My Drive.

**Inputs:** `data/product_metadata_clean.parquet`, `data/train_sampled.parquet`, `val.parquet`, `test.parquet` (from `step1_data_preprocess.ipynb`), and `features/model1/features_{train,val,test}.parquet` (from `step2_model1_feature_engineering.ipynb`).

**Outputs:** `features/model1/features_w2v_{train,val,test}.parquet`; trained models in `data/w2v/`.

**Run order:** optional appendix. Run any time after `step2_model1_feature_engineering.ipynb`; no other notebook depends on it.

**Runtime:** training takes about 3–4 hours on Colab CPUs. Outputs from a full run are preserved below, so it does not need rerunning to see the results.

**Result:** word2vec does not improve on TF-IDF inside the model (see section 9), so the final Model 1 uses TF-IDF similarity only.

## 0. Setup

In [ ]:
# gensim is not preinstalled in Colab. If Colab asks to restart the runtime after installing, restart and run all again.
%pip install -q gensim

In [1]:
import os, sys, io, time, logging, contextlib, threading
from pathlib import Path
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.compute as pc, pyarrow.parquet as pq
from scipy import sparse
from gensim.models import Word2Vec

# logging.basicConfig(format="%(asctime)s %(message)s", level=logging.INFO)   # uncomment for gensim progress

# Project folders (same in every notebook)
from google.colab import drive
drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/Grp8Submission")
DATA = PROJECT / "data"
FEAT_M1 = PROJECT / "features" / "model1"
FEAT_M2 = PROJECT / "features" / "model2"
for d in (DATA, FEAT_M1, FEAT_M2):
    d.mkdir(parents=True, exist_ok=True)
FEAT = FEAT_M1

W2V_DIR = DATA / "w2v"                    # trained models (kept on Drive, reloaded on reruns)
W2V_DIR.mkdir(parents=True, exist_ok=True)
CATALOGUE = DATA / "product_metadata_clean.parquet"
CORPUS_DIR = Path("/content/w2v_corpus")  # 3 GB corpus: Colab's local disk (writing it to Drive is slow)
CORPUS_DIR.mkdir(parents=True, exist_ok=True)
CORPUS_FULL = CORPUS_DIR / "corpus_full.txt"
CORPUS_SAMPLE = CORPUS_DIR / "corpus_sample.txt"

SEED = 42
SAMPLE_N = 3_000_000            # names in the tuning sample
N_BUCKETS = 64                  # temporary files used to shuffle the corpus without loading it all
WORKERS = max(1, (os.cpu_count() or 2) - 1)

BASE = dict(vector_size=100, sg=1, negative=5, epochs=5, sample=1e-3)   # skip-gram, negative sampling
TUNE_WINDOWS = [10, 20, 40]     # gensim window = terms on EACH side; median name has 27 terms (EDA A2)
TUNE_MIN_COUNT = 5              # lower than the final value: rare terms are rarer in a 25% sample
FINAL_MIN_COUNT = 10            # EDA A5/A6: drops 0.34% of tokens, barely changes val OOV
FORCE_RETRAIN = False

SCAN_BATCH = 1_000_000          # same batch size as the main notebook's catalogue scan ...
PROFILE_PER_BATCH, PROFILE_CAP = 3, 200   # ... so category vectors use the same products as the TF-IDF profile
COHERENCE_PER_CATE = 20         # products per category kept for the coherence check

TFIDF = ["query_candidate_name_sim", "query_candidate_cate4_sim"]
W2V = ["w2v_name_sim", "w2v_cate4_sim"]


def load(path, columns=None):
    # ignore_metadata avoids the pandas 2.x/3.x pyarrow-metadata TypeError
    return pq.read_table(path, columns=columns).to_pandas(ignore_metadata=True)


class GensimDotFilter(io.TextIOBase):
    # gensim 4.4.0 on macOS: a dot product of exactly -1.0 is misreported as an error and replaced by 0.0
    # (github.com/piskvorky/gensim/issues/3665). Python prints these straight to stderr (sys.unraisablehook
    # is bypassed), so during training stderr is swapped for this filter: those lines are counted,
    # everything else passes through. Each one affects a single update out of hundreds of millions.
    def __init__(self, real):
        self.real, self.n, self.buf, self.lock = real, 0, "", threading.Lock()

    def write(self, s):
        with self.lock:
            self.buf += s
            *lines, self.buf = self.buf.split("\n")
            for line in lines:
                if "our_dot_float" in line:
                    self.n += 1
                else:
                    self.real.write(line + "\n")
        return len(s)

    def flush(self):
        if self.buf and "our_dot_float" not in self.buf:
            self.real.write(self.buf)
        self.buf = ""
        self.real.flush()


print("workers:", WORKERS)

workers: 14


## 1. Write the corpus (once)
Each product name becomes one line of space-separated term IDs, in the name's original order. Lines are sent to random temporary bucket files, then each bucket is shuffled and appended to the final file. The corpus therefore ends up in random order without holding 12M names in memory, so the first `SAMPLE_N` lines are a uniform random sample. Random order also matters for training itself, because word2vec's learning rate decays over the file and a catalogue sorted by category would bias the vectors.

In [2]:
def write_corpus(catalogue, full_path, sample_path, sample_n, n_buckets=N_BUCKETS, seed=SEED):
    rng = np.random.default_rng(seed)
    bucket_paths = [full_path.with_suffix(f".b{i}") for i in range(n_buckets)]
    buckets = [open(p, "w") for p in bucket_paths]
    n = 0
    for b in pq.ParquetFile(catalogue).iter_batches(batch_size=SCAN_BATCH, columns=["name"]):
        names = b.column("name")
        sep = pa.scalar(" ", type=names.type.value_type)    # separator must match the term type (string / large_string)
        lines = pc.binary_join(names, sep).to_numpy(zero_copy_only=False)
        lines = lines[np.array([isinstance(s, str) and len(s) > 0 for s in lines], dtype=bool)]
        dest = rng.integers(0, n_buckets, len(lines))
        for i in range(n_buckets):
            part = lines[dest == i]
            if len(part):
                buckets[i].write("\n".join(part) + "\n")
        n += len(lines)
    for f in buckets:
        f.close()

    in_sample = 0
    with open(full_path, "w") as full, open(sample_path, "w") as samp:
        for p in bucket_paths:
            lines = p.read_text().splitlines()
            rng.shuffle(lines)
            if lines:
                full.write("\n".join(lines) + "\n")
            take = min(len(lines), sample_n - in_sample)
            if take > 0:
                samp.write("\n".join(lines[:take]) + "\n")
                in_sample += take
            p.unlink()
    return n, in_sample


if CORPUS_FULL.exists() and CORPUS_SAMPLE.exists():
    print("corpus files already exist")
else:
    t0 = time.time()
    n_full, n_sample = write_corpus(CATALOGUE, CORPUS_FULL, CORPUS_SAMPLE, SAMPLE_N)
    print(f"wrote {n_full:,} names (sample {n_sample:,}) in {(time.time() - t0) / 60:.1f} min")
print(f"corpus_full: {CORPUS_FULL.stat().st_size / 1e9:.2f} GB | corpus_sample: {CORPUS_SAMPLE.stat().st_size / 1e9:.2f} GB")

wrote 12,141,247 names (sample 3,000,000) in 0.2 min
corpus_full: 3.02 GB | corpus_sample: 0.75 GB


## 2. Training and text vectors
- **Training** streams the corpus from disk (`corpus_file`), so memory is driven by the vocabulary, not by the 12M names. Multi-threaded training is not bit-for-bit reproducible, so every model is saved and reloaded on reruns. On macOS, gensim 4.4.0 prints a spurious `our_dot_float` warning whenever a dot product is exactly -1.0 (gensim issue #3665); these are counted and hidden during training (see `GensimDotFilter` in section 0), and the count is printed with each model.
- **Text vector** = IDF-weighted average of the text's known term vectors, scaled to unit length, so cosine similarity is a dot product. IDF uses each term's corpus count: names rarely repeat a term, so this is close to the number of names containing it. Unknown terms are skipped; a text with no known term gets a NaN vector, so its feature is missing (as for products without metadata).

In [3]:
def train_w2v(corpus_path, window, min_count, model_path):
    if model_path.exists() and not FORCE_RETRAIN:
        print("loading", model_path.name)
        return Word2Vec.load(str(model_path))
    t0 = time.time()
    dot_filter = GensimDotFilter(sys.stderr)
    with contextlib.redirect_stderr(dot_filter):
        model = Word2Vec(corpus_file=str(corpus_path), window=window, min_count=min_count,
                         workers=WORKERS, seed=SEED, **BASE)
    dot_filter.flush()
    model.save(str(model_path))
    print(f"trained {model_path.name}: {len(model.wv):,} terms in {(time.time() - t0) / 60:.1f} min "
          f"| gensim dot-product warnings suppressed: {dot_filter.n:,}")
    return model


class TermWeights:
    # vocabulary lookup, raw term vectors and IDF weights of one trained model
    def __init__(self, model):
        kv = model.wv
        self.index = pd.Index(kv.index_to_key)
        self.vecs = kv.vectors
        counts = np.array([kv.get_vecattr(k, "count") for k in kv.index_to_key], dtype=np.float64)
        self.idf = np.log((1 + model.corpus_count) / (1 + counts)) + 1


def text_vectors(lists, tw):
    # lists: pyarrow list array of term-ID strings, one text per element -> (n_texts, dim) unit vectors
    if isinstance(lists, pa.ChunkedArray):
        lists = lists.combine_chunks()
    flat = pc.list_flatten(lists).to_numpy(zero_copy_only=False)
    rows = pc.list_parent_indices(lists).to_numpy()
    idx = tw.index.get_indexer(flat)
    ok = idx >= 0
    m = sparse.csr_matrix((tw.idf[idx[ok]], (rows[ok], idx[ok])), shape=(len(lists), len(tw.idf)))
    out = np.asarray(m @ tw.vecs, dtype=np.float32)
    norm = np.linalg.norm(out, axis=1)
    out[norm > 0] /= norm[norm > 0, None]
    out[norm == 0] = np.nan
    return out


def rowwise_cos(A, ia, B, ib, chunk=1_000_000):
    # cos(A[ia[k]], B[ib[k]]) for every k (rows are unit length); index -1 or a NaN vector -> NaN
    out = np.full(len(ia), np.nan, dtype=np.float32)
    for s in range(0, len(ia), chunk):
        a, b = ia[s:s + chunk], ib[s:s + chunk]
        ok = (a >= 0) & (b >= 0)
        out[s:s + chunk][ok] = np.einsum("ij,ij->i", A[a[ok]], B[b[ok]])
    return out

## 3. Catalogue scan: name vectors and category vectors
One pass over the catalogue per model and split:
- **Name vectors** for the split's candidate products.
- **Category vectors**: the average name vector of the same product sample the TF-IDF category profile uses (3 products per category per 1M-row batch, at most 200 per category), so the two `cate4` features differ only in how similarity is measured.
- A few products per category for the **coherence check** (section 5).

In [4]:
def scan_catalogue(tw, need_wids):
    need = np.unique(np.asarray(need_wids, dtype=np.int64))
    need_arr = pa.array(need)
    dim = tw.vecs.shape[1]
    name_vec = np.full((len(need), dim), np.nan, dtype=np.float32)
    name_cate = np.full(len(need), -1, dtype=np.int64)
    taken, sums, counts = {}, {}, {}
    coh_vec, coh_cate = [], []

    for b in pq.ParquetFile(CATALOGUE).iter_batches(batch_size=SCAN_BATCH, columns=["wid", "name", "cate_id_4"]):
        wid = b.column("wid").to_numpy()
        c4 = b.column("cate_id_4").to_numpy()
        is_need = pc.is_in(b.column("wid"), value_set=need_arr).to_numpy(zero_copy_only=False)

        # category profile: first PROFILE_PER_BATCH products per category in this batch, PROFILE_CAP overall
        valid = np.flatnonzero(c4 >= 0)
        first = valid[pd.Series(c4[valid]).groupby(c4[valid], sort=False).cumcount().to_numpy() < PROFILE_PER_BATCH]
        before = np.array([taken.get(x, 0) for x in c4[first]], dtype=np.int64)
        rank = pd.Series(c4[first]).groupby(c4[first], sort=False).cumcount().to_numpy()
        keep = before + rank < PROFILE_CAP
        prof, prof_order = first[keep], (before + rank)[keep]
        for x, k in zip(*np.unique(c4[prof], return_counts=True)):
            taken[x] = taken.get(x, 0) + k
        in_prof = np.zeros(len(wid), bool); in_prof[prof] = True
        in_coh = np.zeros(len(wid), bool); in_coh[prof[prof_order < COHERENCE_PER_CATE]] = True

        sel = np.flatnonzero(is_need | in_prof)
        if len(sel) == 0:
            continue
        v = text_vectors(b.column("name").take(pa.array(sel)), tw)
        has_vec = ~np.isnan(v[:, 0])

        m = is_need[sel]                                    # candidate products
        pos = np.searchsorted(need, wid[sel][m])
        name_vec[pos], name_cate[pos] = v[m], c4[sel][m]

        p = in_prof[sel] & has_vec                          # category vectors
        if p.any():
            u, inv = np.unique(c4[sel][p], return_inverse=True)
            s = np.zeros((len(u), dim)); np.add.at(s, inv, v[p])
            for x, sx, kx in zip(u, s, np.bincount(inv)):
                sums[x] = sums.get(x, 0) + sx
                counts[x] = counts.get(x, 0) + kx

        q = in_coh[sel] & has_vec                           # coherence sample
        coh_vec.append(v[q]); coh_cate.append(c4[sel][q])

    ids = np.array(sorted(sums), dtype=np.int64)
    cv = np.stack([sums[x] / counts[x] for x in ids]).astype(np.float32)
    norm = np.linalg.norm(cv, axis=1)
    cv[norm > 0] /= norm[norm > 0, None]
    cv[norm == 0] = np.nan
    return {"names": {"wid": need, "vec": name_vec, "cate": name_cate},
            "cates": {"id": ids, "vec": cv},
            "coh": {"vec": np.concatenate(coh_vec), "cate": np.concatenate(coh_cate)}}

## 4. Build the features for one split
Candidate rows come from the main feature file, so the output has exactly the same rows in the same order (and the label and TF-IDF features are available for evaluation). Queries come from the split's slate file. Candidates flagged as having no metadata get NaN, the same rule as the main notebook.

In [5]:
CAND_COLS = ["slate_id", "wid", "label", "candidate_has_metadata"] + TFIDF


def build_split(split, tw):
    cand = load(FEAT / f"features_{split}.parquet", columns=CAND_COLS)
    slates = pq.read_table(DATA / f"{'train_sampled' if split == 'train' else split}.parquet",
                           columns=["slate_id", "query"])
    queries = slates.column("query").combine_chunks()
    qv = text_vectors(queries, tw)
    qi = pd.Index(slates.column("slate_id").to_numpy()).get_indexer(cand["slate_id"].to_numpy())
    assert (qi >= 0).all(), "some candidate slates have no query"

    scan = scan_catalogue(tw, cand["wid"].to_numpy())
    names, cates = scan["names"], scan["cates"]
    ni = pd.Index(names["wid"]).get_indexer(cand["wid"].to_numpy())
    cand["w2v_name_sim"] = rowwise_cos(qv, qi, names["vec"], ni)
    ci = pd.Index(cates["id"]).get_indexer(np.where(ni >= 0, names["cate"][np.maximum(ni, 0)], -1))
    cand["w2v_cate4_sim"] = rowwise_cos(qv, qi, cates["vec"], ci)
    no_meta = cand["candidate_has_metadata"].eq(0).to_numpy()
    cand.loc[no_meta, W2V] = np.nan

    # out-of-vocabulary rates (compare with the w2v EDA, A6)
    flat = pc.list_flatten(queries).to_numpy(zero_copy_only=False)
    name_missing = np.isnan(names["vec"][np.maximum(ni, 0), 0]) | (ni < 0)
    stats = {"query_token_oov_%": 100 * (tw.index.get_indexer(flat) < 0).mean(),
             "queries_all_oov_%": 100 * np.isnan(qv[:, 0]).mean(),
             "cands_with_meta_but_no_name_vector_%": 100 * name_missing[~no_meta].mean()}
    return cand, scan, {k: round(float(v), 3) for k, v in stats.items()}

## 5. Evaluation helpers
**Per-slate metrics:** the same code as section 9.6 of `step2_model1_feature_engineering.ipynb`, copied unchanged so every feature is scored identically. Candidates are ranked within each val slate; metrics are averaged over slates with a purchase. `random` (constant score) is the floor.

**Coherence check:** term IDs are anonymised, so we can't inspect nearest neighbours. Instead, products in the **same** L4 category should have more similar name vectors than products from **different** categories. `coherence_auc` is the probability that a random same-category pair is more similar than a random different-category pair (0.5 = no structure).

In [6]:
METRICS = ["ndcg@10", "hit@10", "hit@1", "mrr"]


def _gains(label, relevance):
    label = np.asarray(label)
    if relevance == "purchase":
        return (label == 3).astype(np.float64)
    if relevance == "graded":
        return (2.0 ** label - 1.0).astype(np.float64)
    raise ValueError("relevance must be 'purchase' or 'graded'")


def _group_layout(sorted_slates):
    n = len(sorted_slates)
    start = np.r_[0, np.flatnonzero(sorted_slates[1:] != sorted_slates[:-1]) + 1]
    size = np.diff(np.r_[start, n])
    rank = np.arange(n) - np.repeat(start, size) + 1
    return start, rank


def per_slate_metrics(df, score_col, k=10, relevance="purchase", higher_is_better=True,
                      n_tiebreaks=5, seed=0):
    slate = df["slate_id"].to_numpy()
    gain = _gains(df["label"].to_numpy(), relevance)
    has_rel = pd.Series(gain > 0).groupby(slate).transform("any").to_numpy()
    slate, gain = slate[has_rel], gain[has_rel]
    score = df[score_col].to_numpy(dtype=np.float64)[has_rel]
    if len(slate) == 0:
        raise ValueError("no slates with a relevant item")
    key = score if higher_is_better else -score
    key = np.where(np.isnan(key), -np.inf, key)
    ideal_order = np.lexsort((-gain, slate))
    start, rank = _group_layout(slate[ideal_order])
    disc = np.where(rank <= k, 1.0 / np.log2(rank + 1), 0.0)
    idcg = np.add.reduceat(gain[ideal_order] * disc, start)
    rng = np.random.default_rng(seed)
    acc = {m: np.zeros(len(start)) for m in METRICS}
    for _ in range(n_tiebreaks):
        tie = rng.random(len(slate))
        order = np.lexsort((tie, -key, slate))
        g = gain[order]
        rel = g > 0
        acc["ndcg@10"] += np.add.reduceat(g * disc, start) / idcg
        acc["hit@10"] += np.add.reduceat((rel & (rank <= k)).astype(float), start) > 0
        acc["hit@1"] += np.add.reduceat((rel & (rank == 1)).astype(float), start) > 0
        first = np.minimum.reduceat(np.where(rel, rank, np.inf), start)
        acc["mrr"] += 1.0 / first
    out = pd.DataFrame({m: v / n_tiebreaks for m, v in acc.items()})
    out = out.rename(columns={"ndcg@10": f"ndcg@{k}", "hit@10": f"hit@{k}"})
    out.insert(0, "slate_id", slate[ideal_order][start])
    return out


def summarize(per_slate, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    cols = [c for c in per_slate.columns if c != "slate_id"]
    vals = per_slate[cols].to_numpy()
    idx = rng.integers(0, len(vals), size=(n_boot, len(vals)))
    boot = vals[idx].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"mean": vals.mean(axis=0), "ci_low": lo, "ci_high": hi}, index=cols)


def evaluate_features(df, cols, lower_is_better=(), k=10, relevance="purchase", n_tiebreaks=5, seed=0):
    tmp = df[["slate_id", "label"]].copy()
    tmp["__random__"] = 0.0
    rows = {}
    for c in ["__random__"] + list(cols):
        tmp["__score__"] = df[c].to_numpy() if c != "__random__" else tmp["__random__"].to_numpy()
        ps = per_slate_metrics(tmp, "__score__", k=k, relevance=relevance,
                               higher_is_better=c not in set(lower_is_better),
                               n_tiebreaks=n_tiebreaks, seed=seed)
        rows["random" if c == "__random__" else c] = ps.drop(columns="slate_id").mean()
    return pd.DataFrame(rows).T


def coherence(coh, n_pairs=200_000, seed=SEED):
    v, c = coh["vec"], coh["cate"]
    rng = np.random.default_rng(seed)
    i, j = rng.integers(0, len(v), n_pairs), rng.integers(0, len(v), n_pairs)
    d = c[i] != c[j]
    cos_diff = np.einsum("ij,ij->i", v[i[d]], v[j[d]])
    order = np.argsort(c, kind="stable"); cs = c[order]
    start, size = np.searchsorted(cs, cs, "left"), np.searchsorted(cs, cs, "right") - np.searchsorted(cs, cs, "left")
    k = rng.integers(0, len(v), n_pairs); k = k[size[k] > 1]
    other = start[k] + rng.integers(0, size[k] - 1)
    other = np.where(other >= k, other + 1, other)          # a different product from the same category
    cos_same = np.einsum("ij,ij->i", v[order[k]], v[order[other]])
    n = min(len(cos_same), len(cos_diff))
    return {"cos_same_cate": round(float(cos_same.mean()), 4), "cos_diff_cate": round(float(cos_diff.mean()), 4),
            "coherence_auc": round(float((cos_same[:n] > cos_diff[:n]).mean()), 4)}

## 6. Tuning on the sample
Each `window` setting is trained on the same `SAMPLE_N`-name sample, so differences come from the setting alone. For each model: coherence, OOV rates, and per-slate metrics of both features on **val**. Test is never used for choices.

In [7]:
tune_rows, tune_val = [], {}
for w in TUNE_WINDOWS:
    model = train_w2v(CORPUS_SAMPLE, w, TUNE_MIN_COUNT, W2V_DIR / f"w2v_sample_w{w}_mc{TUNE_MIN_COUNT}.model")
    tw = TermWeights(model)
    t0 = time.time()
    val, scan, stats = build_split("val", tw)
    res = evaluate_features(val, W2V)
    tune_rows.append({"window": w, "vocab": len(tw.index), **coherence(scan["coh"]), **stats,
                      **{f"{c}_ndcg@10": res.loc[c, "ndcg@10"] for c in W2V},
                      **{f"{c}_hit@10": res.loc[c, "hit@10"] for c in W2V}})
    tune_val[w] = val[W2V].copy()
    print(f"window {w}: features + evaluation in {(time.time() - t0) / 60:.1f} min")
    del model, tw, scan

tune = pd.DataFrame(tune_rows).set_index("window")
reference = evaluate_features(val, TFIDF)          # TF-IDF and random on the same val slates
print(tune.round(4).T.to_string())
print("\nreference (TF-IDF features and random):")
print(reference.round(4).to_string())

trained w2v_sample_w10_mc5.model: 210,147 terms in 3.8 min | gensim dot-product warnings suppressed: 309
window 10: features + evaluation in 0.2 min
trained w2v_sample_w20_mc5.model: 210,147 terms in 6.1 min | gensim dot-product warnings suppressed: 436
window 20: features + evaluation in 0.2 min
trained w2v_sample_w40_mc5.model: 210,147 terms in 8.0 min | gensim dot-product warnings suppressed: 598
window 40: features + evaluation in 0.2 min
window                                         10           20           40
vocab                                 210147.0000  210147.0000  210147.0000
cos_same_cate                              0.8426       0.8402       0.8388
cos_diff_cate                              0.4456       0.4330       0.4288
coherence_auc                              0.9794       0.9796       0.9794
query_token_oov_%                          1.2860       1.2860       1.2860
queries_all_oov_%                          0.6030       0.6030       0.6030
cands_with_meta_but_n

**Choosing the window:** highest mean NDCG@10 of the two word2vec features on val. Check the table before accepting it: if the settings are within each other's noise, prefer the smaller window (faster to train on the full catalogue). Set `BEST_WINDOW` by hand to override.

In [8]:
score = tune[[f"{c}_ndcg@10" for c in W2V]].mean(axis=1)
BEST_WINDOW = int(score.idxmax())
print("mean NDCG@10 by window:", score.round(4).to_dict(), "-> BEST_WINDOW =", BEST_WINDOW)

mean NDCG@10 by window: {10: 0.0768, 20: 0.0754, 40: 0.0749} -> BEST_WINDOW = 10


## 7. Final model on the full catalogue
The chosen window is retrained on all names with `FINAL_MIN_COUNT`, then both features are built for train, val and test and saved. The row order matches the main feature files, so they can be joined on `(slate_id, wid)` or by position.

In [9]:
model = train_w2v(CORPUS_FULL, BEST_WINDOW, FINAL_MIN_COUNT,
                  W2V_DIR / f"w2v_full_w{BEST_WINDOW}_mc{FINAL_MIN_COUNT}.model")
tw = TermWeights(model)
print(f"vocabulary: {len(tw.index):,} terms")

trained w2v_full_w10_mc10.model: 276,268 terms in 16.3 min | gensim dot-product warnings suppressed: 1,150
vocabulary: 276,268 terms


In [10]:
final, final_stats = {}, {}
for split in ["train", "val", "test"]:
    t0 = time.time()
    f, scan, stats = build_split(split, tw)
    f[["slate_id", "wid"] + W2V].to_parquet(FEAT / f"features_w2v_{split}.parquet", index=False)
    final[split], final_stats[split] = f, stats
    if split == "val":
        coh_full = coherence(scan["coh"])
    print(f"{split}: {len(f):,} rows in {(time.time() - t0) / 60:.1f} min | {stats}")
    del scan
print("coherence (full model):", coh_full)

train: 2,988,641 rows in 0.3 min | {'query_token_oov_%': 0.94, 'queries_all_oov_%': 0.479, 'cands_with_meta_but_no_name_vector_%': 0.001}
val: 1,511,179 rows in 0.2 min | {'query_token_oov_%': 0.959, 'queries_all_oov_%': 0.443, 'cands_with_meta_but_no_name_vector_%': 0.0}
test: 1,500,706 rows in 0.2 min | {'query_token_oov_%': 0.881, 'queries_all_oov_%': 0.405, 'cands_with_meta_but_no_name_vector_%': 0.0}
coherence (full model): {'cos_same_cate': 0.8311, 'cos_diff_cate': 0.4131, 'coherence_auc': 0.9799}


### 7.1 Sanity checks

In [11]:
for split, f in final.items():
    main = load(FEAT / f"features_{split}.parquet", columns=["slate_id", "wid"])
    assert (main["slate_id"].to_numpy() == f["slate_id"].to_numpy()).all() and \
           (main["wid"].to_numpy() == f["wid"].to_numpy()).all(), f"{split}: rows not aligned"
    for c in W2V:
        x = f[c].dropna()
        assert x.between(-1.0001, 1.0001).all(), f"{split}: {c} outside [-1, 1]"
    # missing values should be close to the share of candidates without metadata (~0.45%)
    print(split, "missing %:", {c: round(100 * f[c].isna().mean(), 3) for c in W2V},
          "| no-metadata %:", round(100 * f["candidate_has_metadata"].eq(0).mean(), 3))
print("rows aligned with the main feature files")

train missing %: {'w2v_name_sim': np.float64(0.903), 'w2v_cate4_sim': np.float64(0.902)} | no-metadata %: 0.454
val missing %: {'w2v_name_sim': np.float64(0.78), 'w2v_cate4_sim': np.float64(0.779)} | no-metadata %: 0.449
test missing %: {'w2v_name_sim': np.float64(0.762), 'w2v_cate4_sim': np.float64(0.762)} | no-metadata %: 0.459
rows aligned with the main feature files


## 8. Evaluation on val
### 8.1 Per-slate metrics: word2vec (full and sample) vs TF-IDF

In [12]:
v = final["val"].copy()
best = tune_val[BEST_WINDOW]
v["w2v_name_sim_sample"], v["w2v_cate4_sim_sample"] = best["w2v_name_sim"].to_numpy(), best["w2v_cate4_sim"].to_numpy()
cols = TFIDF + W2V + ["w2v_name_sim_sample", "w2v_cate4_sim_sample"]
table = evaluate_features(v, cols)
print("val slates with a purchase:", v.loc[v["label"].eq(3), "slate_id"].nunique())
print(table.sort_values("ndcg@10", ascending=False).round(4).to_string())

val slates with a purchase: 1930
                           ndcg@10  hit@10   hit@1     mrr
w2v_name_sim_sample         0.0817  0.1711  0.0238  0.0810
w2v_name_sim                0.0792  0.1706  0.0192  0.0781
query_candidate_name_sim    0.0729  0.1537  0.0212  0.0739
w2v_cate4_sim_sample        0.0719  0.1561  0.0176  0.0729
w2v_cate4_sim               0.0718  0.1553  0.0177  0.0733
query_candidate_cate4_sim   0.0709  0.1522  0.0174  0.0724
random                      0.0667  0.1411  0.0180  0.0693


### 8.2 Bootstrap confidence intervals (95%, over slates)

In [13]:
v["_random"] = 0.0
rows = {}
for c in ["_random"] + TFIDF + W2V:
    s = summarize(per_slate_metrics(v, c))
    rows["random" if c == "_random" else c] = {f"{m} {k}": s.loc[m, k] for m in ["ndcg@10", "hit@10"]
                                               for k in ["mean", "ci_low", "ci_high"]}
print(pd.DataFrame(rows).T.round(4).to_string())

                           ndcg@10 mean  ndcg@10 ci_low  ndcg@10 ci_high  hit@10 mean  hit@10 ci_low  hit@10 ci_high
random                           0.0667          0.0617           0.0714       0.1411         0.1323          0.1491
query_candidate_name_sim         0.0729          0.0640           0.0816       0.1537         0.1378          0.1702
query_candidate_cate4_sim        0.0709          0.0653           0.0767       0.1522         0.1416          0.1625
w2v_name_sim                     0.0792          0.0713           0.0875       0.1706         0.1555          0.1864
w2v_cate4_sim                    0.0718          0.0659           0.0775       0.1553         0.1449          0.1655


### 8.3 Where TF-IDF has nothing to match
Slates where the purchased item shares no query term with its name (TF-IDF name similarity = 0). TF-IDF ranks that item at the bottom by construction, so this is not a fair head-to-head; it shows whether word2vec recovers signal where exact matching cannot.

In [14]:
zero = v.loc[v["label"].eq(3) & v["query_candidate_name_sim"].eq(0), "slate_id"].unique()
sub = v[v["slate_id"].isin(zero)]
print(f"slates: {len(zero):,} of {v.loc[v['label'].eq(3), 'slate_id'].nunique():,} purchase slates")
print(evaluate_features(sub, TFIDF + W2V).round(4).to_string())

slates: 161 of 1,930 purchase slates
                           ndcg@10  hit@10   hit@1     mrr
random                      0.0614  0.1255  0.0186  0.0664
query_candidate_name_sim    0.0169  0.0360  0.0087  0.0312
query_candidate_cate4_sim   0.0546  0.1217  0.0112  0.0588
w2v_name_sim                0.0354  0.0770  0.0124  0.0465
w2v_cate4_sim               0.0696  0.1516  0.0149  0.0695


### 8.4 Overlap with TF-IDF
High correlation means word2vec mostly repeats TF-IDF. Pooled over candidates and averaged within slates (the within-slate version is what matters for ranking).

In [15]:
pairs = [("query_candidate_name_sim", "w2v_name_sim"), ("query_candidate_cate4_sim", "w2v_cate4_sim")]
s = v.sample(n=min(len(v), 500_000), random_state=SEED)
for a, b in pairs:
    pooled = s[[a, b]].corr(method="spearman").iloc[0, 1]
    ranks = v[["slate_id", a, b]].dropna()
    ranks[[a, b]] = ranks.groupby("slate_id")[[a, b]].rank()
    within = ranks.groupby("slate_id")[[a, b]].corr().xs(a, level=1)[b].mean()
    print(f"{b} vs {a}: Spearman pooled {pooled:.3f} | mean within-slate {within:.3f}")

w2v_name_sim vs query_candidate_name_sim: Spearman pooled 0.545 | mean within-slate 0.392
w2v_cate4_sim vs query_candidate_cate4_sim: Spearman pooled 0.749 | mean within-slate 0.646


### 8.5 Summary

In [16]:
print("chosen window:", BEST_WINDOW, "| final min_count:", FINAL_MIN_COUNT, "| vocabulary:", f"{len(tw.index):,}")
print("coherence, full model:", coh_full)
print("coherence, best sample model:", {k: tune.loc[BEST_WINDOW, k] for k in ["cos_same_cate", "cos_diff_cate", "coherence_auc"]})
print("OOV (val, full model):", final_stats["val"])
print(table.loc[["random"] + cols, ["ndcg@10", "hit@10"]].round(4).to_string())

chosen window: 10 | final min_count: 10 | vocabulary: 276,268
coherence, full model: {'cos_same_cate': 0.8311, 'cos_diff_cate': 0.4131, 'coherence_auc': 0.9799}
coherence, best sample model: {'cos_same_cate': np.float64(0.8426), 'cos_diff_cate': np.float64(0.4456), 'coherence_auc': np.float64(0.9794)}
OOV (val, full model): {'query_token_oov_%': 0.959, 'queries_all_oov_%': 0.443, 'cands_with_meta_but_no_name_vector_%': 0.0}
                           ndcg@10  hit@10
random                      0.0667  0.1411
query_candidate_name_sim    0.0729  0.1537
query_candidate_cate4_sim   0.0709  0.1522
w2v_name_sim                0.0792  0.1706
w2v_cate4_sim               0.0718  0.1553
w2v_name_sim_sample         0.0817  0.1711
w2v_cate4_sim_sample        0.0719  0.1561


## 9. Result inside Model 1 (from the ablation, LightGBM, 3 seeds, val NDCG@10)
The in-model comparison was run in the Model 1 ablation (same reference model, objective and number of trees) before it was moved here, so the ablation no longer depends on this notebook.

| Similarity features | NDCG@10 | Difference vs TF-IDF (95% CI) |
|---|---|---|
| none | 0.3749 | −0.0050 (−0.0110 to +0.0014) |
| TF-IDF (used) | 0.3799 | — |
| word2vec | 0.3797 | −0.0001 (−0.0064 to +0.0064) |
| TF-IDF + word2vec | 0.3812 | +0.0013 (−0.0033 to +0.0060) |

Alone (no model, paired bootstrap over the same val slates), word2vec name similarity beats TF-IDF by +0.0063 NDCG@10 (95% CI −0.0025 to +0.0152), also within noise.

**Conclusion:** word2vec matches TF-IDF and adds nothing on top of it (all intervals include 0), so the final Model 1 uses TF-IDF similarity only. Word2vec does recover some signal where the purchased item shares no query term (section 8.3), but those slates are too few to move the overall metric.